# Phase 1 — `best_agent_base.prompts` 데모

## 이 노트북에서 배울 것

1. **베이스 7섹션** 시스템 프롬프트가 어떻게 조립되는지 (`render(ctx)` 호출 → 정적 7개 + 마커 + 동적부)
2. **정적/동적 분리** 의 의미와 KV 캐시 적중률을 어떻게 보호하는지
3. **register/override** 로 베이스 코드 0줄 수정하면서 도메인 콘텐츠 갈아끼우기
4. **`dangerous_uncached`** 가 왜 `reason` 을 필수로 받는지 (기술 부채 추적)
5. **R-5 마커 충돌 가드** 가 어떻게 작동하는지

---

## 실행 방법

- VS Code Jupyter 확장 사용 시: 우측 상단 커널을 `.venv` 에 연결 (또는 아래 setup 셀이 sys.path 처리)
- 터미널: `uv add --dev jupyter ipykernel && uv run jupyter notebook notebooks/`

## 주의

마지막 **Cleanup 셀** 은 꼭 실행. 그래야 같은 커널 안에서 다른 코드가 영향받지 않음 (registry 가 모듈-레벨 싱글톤이라).

---

In [ ]:
# 0. Setup — 노트북 커널이 .venv 가 아닌 경우 프로젝트 루트를 sys.path 에 추가.
#    .venv 커널을 잘 잡았다면 이 셀은 no-op (안전).
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
print(f"project root on sys.path: {_PROJECT_ROOT}")

## 1부 — 베이스 7섹션 둘러보기

### 1.1 `render(ctx)` 가 만들어내는 시스템 프롬프트

베이스가 제공하는 7섹션이 어떤 모양으로 합쳐지는지 직접 본다.

**핵심 개념**:
- `RenderContext()` = turn 별 동적 데이터를 담는 컨테이너 (Phase 1 에선 빈 모델)
- `render(ctx) → str` = 7섹션 + BOUNDARY 마커 + 동적부 조립
- 베이스 콘텐츠는 **도메인-중립** 한 짧은 산문 (헤더/불릿 없음 — 도메인이 자기 형식 결정)

In [ ]:
from best_agent_base.prompts.boundary import SYSTEM_PROMPT_DYNAMIC_BOUNDARY
from best_agent_base.prompts.render import RenderContext, render, get_static_hash

# RenderContext() = 빈 인스턴스. Phase 1 에선 필드 0개.
# Phase 2+ 가 current_time, tools, attachments 같은 필드를 추가할 슬롯.
ctx = RenderContext()

output = render(ctx)
print(output)

### 1.2 BOUNDARY 마커로 정적·동적 분리 확인

왜 마커를 쓰나?  
Anthropic/OpenAI/Gemini 모두 같은 prefix 가 들어오면 KV 캐시를 재사용한다. **정적 부 = 모든 turn 에서 동일** 이어야 캐시 hit. 그래서 베이스가:

```
<정적 7섹션>      ← 매 turn 동일 (KV 캐시 적중)
__BOUNDARY__
<동적부>          ← 매 turn 변경 가능 (캐시 안 됨)
```

마커 자체는 모델이 무시할 일반 토큰열 — split 용도일 뿐.

In [ ]:
static_part, dynamic_part = output.split(SYSTEM_PROMPT_DYNAMIC_BOUNDARY)

print("=== 정적 부 (앞, 캐시 가능) ===")
print(static_part)
print(f"\n=== 마커 ===\n{SYSTEM_PROMPT_DYNAMIC_BOUNDARY!r}")
print(f"\n=== 동적 부 (뒤, 매번 변경 허용) ===")
# Phase 1 의 베이스에는 동적 섹션이 없음 → 빈 문자열 (앞뒤 \n\n 만 남음)
print(repr(dynamic_part))

### 1.3 `get_static_hash(ctx)` — Phase 2 캐시 측정 슬롯

정적 부 문자열의 **sha256 hex 첫 16자** 를 반환. 같은 ctx + 같은 registry → 항상 같은 hash. 이 hash 가 Phase 2 의 KV 캐시 적중률 측정 키가 된다.

**왜 16자만?** 64비트 = ~10^19 가지 → 캐시 키 용도로 충돌 위험 무시 가능, 로그 짧음.

In [ ]:
h = get_static_hash(ctx)
print(f"static hash = {h}")

# 정적 안정성 검증: N=10 호출 결과가 모두 동일해야 (FR-4 / R-3)
hashes = [get_static_hash(ctx) for _ in range(10)]
print(f"N=10 호출 결과 모두 동일?: {len(set(hashes)) == 1}")
print(f"  → 만약 False 면 정적 섹션에 동적 데이터 (현재시각 등) 가 섞여있다는 뜻.")

---

## 2부 — 도메인이 베이스 갈아끼우기 (D-8 swappable)

### 2.1 `registry.register(name, custom)` — 베이스 코드 0줄 수정

코딩 도메인이 §3 `DoingTasks` 를 자기 색깔로 교체. **PromptSection Protocol** (name·static·render) 만 만족하면 어떤 클래스든 OK — 상속 강제 X (duck typing).

헤더 (`# Doing tasks`) 와 불릿 (`-`) 은 **도메인이 자기 render() 결과에 박는 것** — 베이스는 강제하지 않음 (D-2 "안 만들기").

In [ ]:
from best_agent_base.prompts.registry import registry


class CodingDoingTasks:
    name = "DoingTasks"      # ← registry 키. 베이스의 "DoingTasks" 와 일치 = 교체.
    static = True            # ← 정적 영역에 들어감 (캐시 가능)

    def render(self, ctx):
        # 헤더와 불릿은 여기서 결정. 베이스는 형식 강제 안 함.
        return (
            "# Doing tasks\n"
            "- Use type hints; prefer composition over inheritance.\n"
            "- Write tests first (RED → GREEN → commit).\n"
            "- Keep functions under ~30 lines."
        )


# register — 베이스 prompts/ 모듈은 0줄도 수정 안됨
registry.register("DoingTasks", CodingDoingTasks())

print("=== render 후 정적부 (DoingTasks 만 교체된 모습) ===")
print(render(ctx).split(SYSTEM_PROMPT_DYNAMIC_BOUNDARY)[0])

### 2.2 hash 가 바뀐다 = 캐시가 깨진다

도메인이 정적 섹션을 갈아끼웠으니 (= 정적 부 문자열이 달라졌으니) **hash 도 달라짐**. 이건 정상 — 도메인 콘텐츠 변경 자체는 캐시를 깰 수밖에 없음. 다만 그 이후로는 **새 hash 기준으로 다시 캐시 hit 누적** 가능.

(잘못된 시나리오: 정적 영역 안에 turn 별로 변하는 데이터가 들어가면 매 turn hash 가 바뀜 → 캐시 평생 miss. 그게 R-3 위험.)

In [ ]:
h_after_override = get_static_hash(ctx)
print(f"hash before override = {h}")
print(f"hash after  override = {h_after_override}")
print(f"hash changed? {h != h_after_override}  ← 도메인 정적 콘텐츠 변경은 한 번만 일어남 (배포 시점)")

# 단, override 후 N회 호출은 다시 동일 (정적 안정성 유지)
hashes_after = [get_static_hash(ctx) for _ in range(10)]
print(f"override 후 N=10 hash 동일?: {len(set(hashes_after)) == 1}")

---

## 3부 — 캐시 escape hatch & 동적 섹션

### 3.1 `dangerous_uncached` — 정적 영역에 의도적 캐시 깨기

원칙적으로는 정적 영역 = 동일. 하지만 "이 데이터는 정적 영역 안에 있어야 하지만 매 turn 변한다" 는 예외 케이스가 있을 수 있음 (예: session ID, 인증 토큰 hash 등).

그럴 때 `dangerous_uncached(name, content, reason)` 으로 **명시적·기록되는 형태로** 만 들어갈 수 있게 강제.

**`reason` 이 왜 필수?** (R-4 mitigation)
- 캐시를 의도적으로 깨는 건 기술 부채 → 누가·왜·언제 추적되어야 함
- 빈 문자열 거부 (Pydantic `min_length=1`) — 의미없는 reason 방지
- Phase 11 Hooks 에서 reason 들을 수집·로깅할 슬롯

In [ ]:
from best_agent_base.prompts.boundary import dangerous_uncached
from pydantic import ValidationError

# 정상 사용 — reason 명확히 기록
obj = dangerous_uncached(
    name="current_time",
    content="Current UTC time: 2026-05-03T13:00:00Z",
    reason="static-region 에 시각 박아야 하는 도메인 정책 (Phase 11 hooks 에서 추적 예정)",
)
print(f"정상 호출")
print(f"   name   = {obj.name!r}")
print(f"   reason = {obj.reason!r}")

# reason 빈 문자열 → Pydantic ValidationError
print("\n=== reason='' 시도 ===")
try:
    dangerous_uncached(name="x", content="y", reason="")
except ValidationError as e:
    print(f"거부됨: {e.errors()[0]['msg']}")

### 3.2 동적 섹션 — 정적 hash 는 불변 (D-1 캐시 보호)

**핵심**: `static=False` 인 섹션을 register 하면 그건 BOUNDARY **뒤** 에 가서 동적부가 됨. 그래서 정적 hash 는 절대 영향받지 않음.

이게 D-1 (정적/동적 분리) 의 진짜 가치. **동적 데이터를 베이스 시스템 프롬프트에 넣어도 캐시 안전**.

In [13]:
import time


class DynamicTimestamp:
    name = "DynamicTimestamp"
    static = False             # ← 핵심: 동적 영역으로 분류

    def render(self, ctx):
        return f"[runtime] now = {time.time():.3f}"


registry.register("DynamicTimestamp", DynamicTimestamp())

h_with_dynamic = get_static_hash(ctx)
print(f"동적 섹션 추가 후 정적 hash = {h_with_dynamic}")
print(f"override 직후 hash 와 동일?  = {h_with_dynamic == h_after_override}")
print("  → 동적 섹션은 정적 hash 에 영향 X. 캐시 안전.\n")

print("=== 전체 render 결과 — 동적부에 timestamp 등장 ===")
print(render(ctx))

동적 섹션 추가 후 정적 hash = fdb02d2a4245727e


NameError: name 'h_after_override' is not defined

---

## 4부 — 안전장치

### 4.1 R-5 — BOUNDARY 마커 충돌 가드

도메인이 `__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__` 를 자기 콘텐츠 안에 우연히 포함시키면 split 로직이 깨짐. 그래서 `_render_static`/`_render_dynamic` 가 **콘텐츠에 마커가 있는지 검사 → ValueError**.

이건 의도하지 않은 사고를 방지하는 fail-loud 가드.

In [ ]:
class EvilSection:
    """마커를 우연히 포함하는 섹션 (실수 시뮬레이션)."""

    name = "Intro"
    static = True

    def render(self, ctx):
        # 도메인이 자기 텍스트에 우연히 BOUNDARY 문자열을 박은 상황
        return f"oops {SYSTEM_PROMPT_DYNAMIC_BOUNDARY} more text"


registry.register("Intro", EvilSection())
try:
    render(ctx)
except ValueError as e:
    print(f"가드 작동: {e}")

---

## Cleanup — 베이스 registry 복구 (필수)

`registry` 는 모듈-레벨 싱글톤이라 한 번 register 하면 같은 커널의 다른 코드 (다른 노트북, 테스트) 에 영향 줌.

`importlib.reload` 로 모듈 재실행 → `for _section in BASE_SECTIONS: registry.register(...)` 가 다시 돌아 7개만 등록된 깨끗한 상태로 복구.

In [ ]:
import importlib
from best_agent_base.prompts import registry as registry_module

importlib.reload(registry_module)
from best_agent_base.prompts.registry import registry  # 갱신된 싱글톤 재바인딩

section_count = len(list(registry.all_sections()))
h_final = get_static_hash(RenderContext())

print(f"registry 복구 완료")
print(f"   등록 섹션 수: {section_count}  (= 7 베이스만 — 처음과 동일)")
print(f"   초기 hash    : {h}")
print(f"   복구 후 hash : {h_final}")
print(f"   동일?         : {h == h_final}")

---

## 다음 단계 — Phase 별 연계 포인트

| Phase | 연계 지점 |
|---|---|
| **Phase 2** (캐시 메트릭) | `get_static_hash(ctx)` 를 cache key 로 → KV 캐시 hit/miss 측정 |
| **Phase 4** (도구) | `RenderContext.tools` 필드 추가 → `UsingTools` 섹션이 동적 도구 카탈로그 렌더 |
| **Phase 9** (context) | `RenderContext.attachments` 추가 → 동적 섹션이 첨부 메타 주입 |
| **Phase 11** (hooks) | `dangerous_uncached.reason` 들을 수집·로깅 |
| **Phase 14** (FastAPI) | `SectionRegistry` → ContextVar 기반 per-request 격리 (R-1 재검토) |

---

## 직접 시도해볼 것 (실습)

1. **`Intro` 섹션을 의료 도메인으로 override** — `name="Intro"`, `render()` 가 "You are a medical advisory agent..." 반환하도록
2. **베이스 7섹션 중 §6 ToneStyle 만 한국어 응답 강제** 로 갈아끼우기
3. **`dangerous_uncached` 로 session ID** 를 정적 영역에 넣어보기 — reason 이 logging 에 어떻게 쓰일지 상상
4. **`MedicalRenderContext(RenderContext)`** 처럼 도메인 ctx 확장 — 추가 필드를 섹션이 어떻게 쓰는지

각 실습 후 마지막 Cleanup 셀 다시 실행.